# 1次元 vs 2次元: CEVAEはどちらでX-learnerに勝てるか

`run_1d_experiment.ipynb` と `run_2d_experiment.ipynb` を実行した結果(`results/*.csv`)を読み込んで、
「潜在交絡因子Zが1次元なら(崩壊を除けば)CEVAEがX-learnerに勝てるが、2次元になると勝てなくなる」
という発見を一つの表・図にまとめる。

(全体ゼミ発表のストーリーライン `CEVAE実験/全体ゼミ_次回_ストーリーライン.md` 第2章・第5章で使う数値)

**事前に `run_1d_experiment.ipynb` と `run_2d_experiment.ipynb` を実行して
`results/1d_main_results.csv` と `results/2d_main_results.csv` を作っておくこと。**

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from diagnostics import guard_threshold

df_1d = pd.read_csv("results/1d_main_results.csv")
df_2d = pd.read_csv("results/2d_main_results.csv")

print("1次元実験:", df_1d.shape, " 2次元実験:", df_2d.shape)

## 1次元: 崩壊を除かない場合の素の成績(全data_seed)

In [ ]:
cevae_1d_mean = df_1d["cevae_pehe"].mean()
cevae_1d_std = df_1d["cevae_pehe"].std()
xl_1d_mean = df_1d["xl_pehe"].mean()
xl_1d_std = df_1d["xl_pehe"].std()
win_1d = int((df_1d["cevae_pehe"] < df_1d["xl_pehe"]).sum())
n_1d = len(df_1d)

print(f"CEVAE √PEHE: {cevae_1d_mean:.3f}±{cevae_1d_std:.3f}")
print(f"X-learner √PEHE: {xl_1d_mean:.3f}±{xl_1d_std:.3f}")
print(f"CEVAE勝率: {win_1d}/{n_1d}")

## 2次元: 崩壊ガード適用後の成績(seedごとにmodel_seedの平均を取って比較)

In [ ]:
guard_th_2d = guard_threshold(df_2d["mse_ratio"])
df_2d_pass = df_2d[df_2d["mse_ratio"] <= guard_th_2d]

cevae_2d_mean = df_2d_pass["sqrt_pehe"].mean()
cevae_2d_std = df_2d_pass["sqrt_pehe"].std()

xl_2d_summary = df_2d.drop_duplicates("data_seed").set_index("data_seed")["xl_pehe"]
xl_2d_mean = xl_2d_summary.mean()
xl_2d_std = xl_2d_summary.std()

per_seed_cevae_2d = df_2d_pass.groupby("data_seed")["sqrt_pehe"].mean()
common_seeds_2d = per_seed_cevae_2d.index.intersection(xl_2d_summary.index)
win_2d = int((per_seed_cevae_2d.loc[common_seeds_2d] < xl_2d_summary.loc[common_seeds_2d]).sum())
n_2d = len(common_seeds_2d)

print(f"崩壊ガード閾値(mse_ratio) = {guard_th_2d:.3f}")
print(f"CEVAE √PEHE(ガード後): {cevae_2d_mean:.3f}±{cevae_2d_std:.3f}")
print(f"X-learner √PEHE: {xl_2d_mean:.3f}±{xl_2d_std:.3f}")
print(f"CEVAE勝率(ガード後): {win_2d}/{n_2d}")

## まとめ表(発表スライド用)

In [ ]:
summary_table = pd.DataFrame({
    "指標": ["CEVAE √PEHE (mean±std)", "X-learner √PEHE (mean±std)", "CEVAE勝率"],
    "1次元Z": [
        f"{cevae_1d_mean:.3f}±{cevae_1d_std:.3f}",
        f"{xl_1d_mean:.3f}±{xl_1d_std:.3f}",
        f"{win_1d}/{n_1d}",
    ],
    "2次元Z(崩壊ガード後)": [
        f"{cevae_2d_mean:.3f}±{cevae_2d_std:.3f}",
        f"{xl_2d_mean:.3f}±{xl_2d_std:.3f}",
        f"{win_2d}/{n_2d}",
    ],
})
summary_table

## 図: data_seedごとの√PEHE比較(1次元 vs 2次元)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=False)

axes[0].scatter(df_1d["data_seed"], df_1d["xl_pehe"], label="X-learner", marker="o")
axes[0].scatter(df_1d["data_seed"], df_1d["cevae_pehe"], label="CEVAE", marker="x")
axes[0].set_title(f"1次元Z (CEVAE勝率 {win_1d}/{n_1d})")
axes[0].set_xlabel("data_seed")
axes[0].set_ylabel("√PEHE")
axes[0].legend()

axes[1].scatter(xl_2d_summary.index, xl_2d_summary.values, label="X-learner", marker="o")
axes[1].scatter(per_seed_cevae_2d.index, per_seed_cevae_2d.values, label="CEVAE(ガード後平均)", marker="x")
axes[1].set_title(f"2次元Z (CEVAE勝率 {win_2d}/{n_2d})")
axes[1].set_xlabel("data_seed")
axes[1].legend()

plt.tight_layout()
plt.show()